# Se liga na música: o dia da semana muda o jeito que as pessoas ouvem música?

# Conteúdo <a id='back'></a>

* [Introdução](#intro)
* [Etapa 1. Visão geral dos dados](#data_review)
    * [O que já dá pra ver](#data_review_conclusions)
* [Etapa 2. Limpeza dos dados](#data_preprocessing)
    * [2.1 Nomes das colunas](#header_style)
    * [2.2 Valores ausentes](#missing_values)
    * [2.3 Duplicados](#duplicates)
    * [2.4 Resumo da limpeza](#data_preprocessing_conclusions)
* [Etapa 3. Teste da hipótese](#hypothesis)
    * [3.1 Atividade dos usuários nas duas cidades](#activity)
* [Conclusões](#end)

## Introdução <a id='intro'></a>

Decisão boa vem de hipótese testada, não de achismo. Aqui eu peguei dados de um serviço de streaming de música e testei uma pergunta simples:

**A atividade dos usuários muda conforme o dia da semana e a cidade?**

As cidades são Springfield e Shelbyville, e os dias analisados são segunda, quarta e sexta.

Na prática, esse tipo de resposta ajuda a decidir **quando** e **onde** concentrar campanha, lançamento ou promoção.

### Como fiz
1. **Visão geral:** entender a base e achar os problemas.
2. **Limpeza:** padronizar colunas, tratar ausentes e remover duplicados.
3. **Teste da hipótese:** comparar o volume de reproduções por cidade e por dia.

Base: `music_project_en.csv`, sem documentação de qualidade, então a limpeza veio antes de qualquer conclusão.

[Voltar ao Índice](#back)

## Etapa 1. Visão geral dos dados <a id='data_review'></a>

Primeiro passo: abrir a base e entender o que tem nela.

In [ ]:
import pandas as  pd  # importando pandas


Carreguei o arquivo e olhei as 10 primeiras linhas:

In [ ]:
# Lendo o arquivo e exibindo as 10 primeiras linhas
df = pd.read_csv("/datasets/music_project_en.csv")
df.head(10)


,userID,Track,artist,genre,City,time,Day
0,FFB692EC,Kamigata To Boots,The Mass Missile,rock,Shelbyville,20:28:33,Wednesday
1,55204538,Delayed Because of Accident,Andreas Rönnberg,rock,Springfield,14:07:09,Friday
2,20EC38,Funiculì funiculà,Mario Lanza,pop,Shelbyville,20:58:07,Wednesday
3,A3DD03C9,Dragons in the Sunset,Fire + Ice,folk,Shelbyville,08:37:09,Monday
4,E2DC1FAE,Soul People,Space Echo,dance,Springfield,08:34:34,Monday
5,842029A1,Chains,Obladaet,rusrap,Shelbyville,13:09:41,Friday
6,4CB90AA5,True,Roman Messer,dance,Springfield,13:00:07,Wednesday
7,F03E1C1F,Feeling This Way,Polina Griffith,dance,Springfield,20:47:49,Wednesday
8,8FA1D3BE,L’estate,Julia Dalia,ruspop,Springfield,09:17:40,Friday
9,E772D5C0,Pessimist,NaN,dance,Shelbyville,21:20:49,Wednesday


Agora a visão geral: tipo de dado e quantos valores preenchidos em cada coluna.

In [ ]:
df.info()  # visão geral das colunas


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65079 entries, 0 to 65078
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0     userID  65079 non-null  object
 1   Track     63736 non-null  object
 2   artist    57512 non-null  object
 3   genre     63881 non-null  object
 4     City    65079 non-null  object
 5   time      65079 non-null  object
 6   Day       65079 non-null  object
dtypes: object(7)
memory usage: 3.5+ MB


São 65.079 linhas e 7 colunas, todas do tipo `object`:

- `userID` — identificação do usuário
- `Track` — nome da música
- `artist` — artista
- `genre` — gênero
- `City` — cidade do usuário
- `time` — horário em que a música tocou
- `Day` — dia da semana

Logo de cara aparecem problemas nos nomes das colunas:
1. Mistura de maiúsculas e minúsculas (`Track`, `City`, `Day` x `artist`, `genre`, `time`).
2. Espaços sobrando (`  userID`, `  City  `).
3. Nome composto sem separação (`userID` deveria ser `user_id`).

E já dá pra ver valores faltando em `Track`, `artist` e `genre`.

### O que já dá pra ver <a id='data_review_conclusions'></a>

- **Cada linha é uma reprodução:** um usuário ouviu uma música, numa cidade, num dia e horário.
- **Os dados bastam pra hipótese:** tem cidade e dia da semana, que é o que preciso pra comparar.
- **Mas precisam de limpeza antes:** nomes de colunas fora do padrão, valores ausentes e possíveis duplicados. Sem tratar isso, a contagem sai errada.

[Voltar ao Índice](#back)

## Etapa 2. Limpeza dos dados <a id='data_preprocessing'></a>

Antes de contar qualquer coisa, deixei a base confiável. Ordem: nomes das colunas → valores ausentes → duplicados.

### Nomes das colunas <a id='header_style'></a>
Como estão hoje:

In [ ]:
df.columns


Index(['  userID', 'Track', 'artist', 'genre', '  City  ', 'time', 'Day'], dtype='object')

Padrão que vou aplicar: tudo minúsculo, sem espaços e com `snake_case`.

Primeiro, tudo em minúsculo:

In [ ]:
new_col_names = []

for col in df.columns:
    new_col_names.append(col.lower())

df.columns = new_col_names

print(df.columns)# Percorrendo os cabeçalhos e convertendo tudo em minúsculos


Index(['  userid', 'track', 'artist', 'genre', '  city  ', 'time', 'day'], dtype='object')


Depois, tiro os espaços das pontas:

In [ ]:
new_col_names = []

for col in df.columns:
    new_col_names.append(col.strip().lower())

df.columns = new_col_names

print(df.columns)# Percorrendo os cabeçalhos e removendo os espaços


Index(['userid', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='object')


Por fim, `userid` vira `user_id`:

In [ ]:
df = df.rename(columns={'userid': 'user_id'})

print(df.columns)# Renomeando a coluna "userid"


Index(['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='object')


Conferindo:

In [ ]:
print(df.columns) # verificando o resultado: a lista de cabeçalhos


Index(['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='object')


In [ ]:
df

,user_id,track,artist,genre,city,time,day
0,FFB692EC,Kamigata To Boots,The Mass Missile,rock,Shelbyville,20:28:33,Wednesday
1,55204538,Delayed Because of Accident,Andreas Rönnberg,rock,Springfield,14:07:09,Friday
2,20EC38,Funiculì funiculà,Mario Lanza,pop,Shelbyville,20:58:07,Wednesday
3,A3DD03C9,Dragons in the Sunset,Fire + Ice,folk,Shelbyville,08:37:09,Monday
4,E2DC1FAE,Soul People,Space Echo,dance,Springfield,08:34:34,Monday
...,...,...,...,...,...,...,...
65074,729CBB09,My Name,McLean,rnb,Springfield,13:32:28,Wednesday
65075,D08D4A55,Maybe One Day (feat. Black Spade),Blu & Exile,hip,Shelbyville,10:00:00,Monday
65076,C5E3A0D5,Jalopiina,NaN,industrial,Springfield,20:09:26,Friday
65077,321D0506,Freight Train,Chas McDevitt,rock,Springfield,21:43:59,Friday


[Voltar ao Índice](#back)

### Valores ausentes <a id='missing_values'></a>
Quantos valores faltam em cada coluna:

In [ ]:


print(df.isna().sum()) # calculando o número de valores ausentes




user_id       0
track      1343
artist     7567
genre      1198
city          0
time          0
day           0
dtype: int64


Três colunas com falta de dados:

- `track` (1.343) e `artist` (7.567): não atrapalham a hipótese, que é sobre volume de reproduções.
- `genre` (1.198, menos de 2% da base): importaria numa análise de gosto musical, mas aqui o impacto é pequeno.

Não tenho como recuperar a origem desses dados, então preenchi tudo com `'unknown'`. Assim não perco nenhuma reprodução na contagem.

Preenchendo as três colunas:

In [ ]:



columns_to_replace = ['track', 'artist', 'genre']

for col in columns_to_replace:
    df[col] = df[col].fillna('unknown')# percorrendo os cabeçalhos e substituindo valores ausentes por 'unknown'





Conferindo se sobrou algum ausente:

In [ ]:
print(df.isna().sum())# contando os valores ausentes


user_id    0
track      0
artist     0
genre      0
city       0
time       0
day        0
dtype: int64


[Voltar ao Índice](#back)

### Duplicados <a id='duplicates'></a>
Linhas repetidas por completo (duplicados explícitos):

In [ ]:
print(df.duplicated().sum())# contando duplicados explícitos


3826


São 3.826 linhas repetidas, quase 6% da base. Se ficassem, iam inflar a contagem. Removi:

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)  # removendo duplicados explícitos
df

,user_id,track,artist,genre,city,time,day
0,FFB692EC,Kamigata To Boots,The Mass Missile,rock,Shelbyville,20:28:33,Wednesday
1,55204538,Delayed Because of Accident,Andreas Rönnberg,rock,Springfield,14:07:09,Friday
2,20EC38,Funiculì funiculà,Mario Lanza,pop,Shelbyville,20:58:07,Wednesday
3,A3DD03C9,Dragons in the Sunset,Fire + Ice,folk,Shelbyville,08:37:09,Monday
4,E2DC1FAE,Soul People,Space Echo,dance,Springfield,08:34:34,Monday
...,...,...,...,...,...,...,...
65074,729CBB09,My Name,McLean,rnb,Springfield,13:32:28,Wednesday
65075,D08D4A55,Maybe One Day (feat. Black Spade),Blu & Exile,hip,Shelbyville,10:00:00,Monday
65076,C5E3A0D5,Jalopiina,unknown,industrial,Springfield,20:09:26,Friday
65077,321D0506,Freight Train,Chas McDevitt,rock,Springfield,21:43:59,Friday


Conferindo:

In [ ]:
print(df.duplicated().sum())# verificando duplicados novamente



0


Agora os duplicados implícitos: o mesmo gênero escrito de jeitos diferentes. Listei os gêneros únicos em ordem alfabética:

In [ ]:

print(sorted(df['genre'].unique()))# visualizando nomes de gêneros únicos



['acid', 'acoustic', 'action', 'adult', 'africa', 'afrikaans', 'alternative', 'ambient', 'americana', 'animated', 'anime', 'arabesk', 'arabic', 'arena', 'argentinetango', 'art', 'audiobook', 'avantgarde', 'axé', 'baile', 'balkan', 'beats', 'bigroom', 'black', 'bluegrass', 'blues', 'bollywood', 'bossa', 'brazilian', 'breakbeat', 'breaks', 'broadway', 'cantautori', 'cantopop', 'canzone', 'caribbean', 'caucasian', 'celtic', 'chamber', 'children', 'chill', 'chinese', 'choral', 'christian', 'christmas', 'classical', 'classicmetal', 'club', 'colombian', 'comedy', 'conjazz', 'contemporary', 'country', 'cuban', 'dance', 'dancehall', 'dancepop', 'dark', 'death', 'deep', 'deutschrock', 'deutschspr', 'dirty', 'disco', 'dnb', 'documentary', 'downbeat', 'downtempo', 'drum', 'dub', 'dubstep', 'eastern', 'easy', 'electronic', 'electropop', 'emo', 'entehno', 'epicmetal', 'estrada', 'ethnic', 'eurofolk', 'european', 'experimental', 'extrememetal', 'fado', 'film', 'fitness', 'flamenco', 'folk', 'folklor

Na lista aparecem `hip`, `hop` e `hip-hop`, que são o mesmo gênero: `hiphop`.

Criei uma função que troca uma lista de nomes errados pelo nome correto. Assim dá pra reaproveitar se aparecer outro caso:

In [ ]:


def replace_wrong_genres(wrong_genres, correct_genre):
    for wrong_genre in wrong_genres:
        df['genre'] = df['genre'].replace(wrong_genre, correct_genre)
# função para substituir duplicados implícitos



Aplicando nas variações de hip hop:

In [ ]:
duplicates = ['hip', 'hop', 'hip-hop']

replace_wrong_genres(duplicates, 'hiphop') # removendo duplicados implícitos


/tmp/ipykernel_480/509180811.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['genre'] = df['genre'].replace(wrong_genre, correct_genre)


Conferindo a lista de novo:

In [ ]:
print(sorted(df['genre'].unique()))# verificando valores duplicados


['acid', 'acoustic', 'action', 'adult', 'africa', 'afrikaans', 'alternative', 'ambient', 'americana', 'animated', 'anime', 'arabesk', 'arabic', 'arena', 'argentinetango', 'art', 'audiobook', 'avantgarde', 'axé', 'baile', 'balkan', 'beats', 'bigroom', 'black', 'bluegrass', 'blues', 'bollywood', 'bossa', 'brazilian', 'breakbeat', 'breaks', 'broadway', 'cantautori', 'cantopop', 'canzone', 'caribbean', 'caucasian', 'celtic', 'chamber', 'children', 'chill', 'chinese', 'choral', 'christian', 'christmas', 'classical', 'classicmetal', 'club', 'colombian', 'comedy', 'conjazz', 'contemporary', 'country', 'cuban', 'dance', 'dancehall', 'dancepop', 'dark', 'death', 'deep', 'deutschrock', 'deutschspr', 'dirty', 'disco', 'dnb', 'documentary', 'downbeat', 'downtempo', 'drum', 'dub', 'dubstep', 'eastern', 'easy', 'electronic', 'electropop', 'emo', 'entehno', 'epicmetal', 'estrada', 'ethnic', 'eurofolk', 'european', 'experimental', 'extrememetal', 'fado', 'film', 'fitness', 'flamenco', 'folk', 'folklor

[Voltar ao Índice](#back)

### Resumo da limpeza <a id='data_preprocessing_conclusions'></a>

- **Colunas** padronizadas: `user_id`, `track`, `artist`, `genre`, `city`, `time`, `day`.
- **Ausentes** preenchidos com `'unknown'`, sem perder nenhuma reprodução.
- **3.826 duplicados explícitos** removidos: a base foi de 65.079 para 61.253 linhas.
- **Duplicados implícitos** de hip hop unificados em `hiphop`.

Base pronta pra análise.

[Voltar ao Índice](#back)

## Etapa 3. Teste da hipótese <a id='hypothesis'></a>

### Atividade dos usuários nas duas cidades <a id='activity'></a>

A hipótese diz que o consumo de música muda conforme a cidade e o dia da semana.

Pra testar, comparei o número de reproduções em Springfield e Shelbyville na segunda, quarta e sexta.

Começo pelo volume total de cada cidade:

In [ ]:
# Conta as músicas tocadas em cada cidade
print(df.groupby('city')['track'].count()) # Contando as músicas tocadas em cada cidade


city
Shelbyville    18512
Springfield    42741
Name: track, dtype: int64


Springfield tem **42.741** reproduções e Shelbyville **18.512**. Ou seja, Springfield concentra cerca de **70%** do consumo.

Isso já mostra que os números absolutos vão sempre favorecer Springfield. Por isso, mais à frente, o que importa é o **padrão** de cada cidade ao longo da semana, não só o volume.

Agora o volume por dia da semana, juntando as duas cidades:

In [ ]:
print(df.groupby('day')['track'].count()) # Calculando as músicas escutadas em cada um desses três dias


day
Friday       21840
Monday       21354
Wednesday    18059
Name: track, dtype: int64


Sexta (21.840) e segunda (21.354) quase empatam, e quarta (18.059) fica abaixo.

Só que esse total junta as duas cidades. Como Springfield pesa 70%, esse padrão pode estar escondendo o que acontece em Shelbyville. Próximo passo: cruzar cidade **e** dia.

Pra cruzar as duas coisas, criei uma função que filtra a base pelo dia e pela cidade e conta as reproduções:

In [ ]:
def number_tracks(day, city):
    # filtra pelo dia
    day_filtered = df[df['day'] == day]

    # filtra pela cidade
    city_filtered = day_filtered[day_filtered['city'] == city]

    # conta as reproduções
    return city_filtered['user_id'].count()

Rodando para as duas cidades nos três dias:

In [ ]:
print(number_tracks('Monday', 'Springfield'))  # Springfield na segunda


In [ ]:
print(number_tracks('Monday', 'Shelbyville'))
# a quantidade de músicas tocadas em Shelbyville na segunda-feira


5614


In [ ]:
print(number_tracks('Wednesday', 'Springfield'))
# a quantidade de músicas tocadas em Springfield na quarta-feira


11056


In [ ]:

print(number_tracks('Wednesday', 'Shelbyville'))
# a quantidade de músicas tocadas em Shelbyville na quarta-feira


7003


In [ ]:
print(number_tracks('Friday', 'Springfield'))
# a quantidade de músicas tocadas em Springfield na sexta-feira


15945


In [ ]:
print(number_tracks('Friday', 'Shelbyville'))# a quantidade de músicas tocadas em Shelbyville na sexta-feira


5895


**Resultado do cruzamento**

| Dia | Springfield | Shelbyville |
|---|---|---|
| Segunda | 15.740 | 5.614 |
| Quarta | 11.056 | 7.003 |
| Sexta | 15.945 | 5.895 |

Aqui aparece a parte mais interessante: **as duas cidades se comportam de forma oposta.**

- **Springfield** ouve mais na segunda e na sexta, e cai na quarta (cerca de 26% da semana).
- **Shelbyville** faz o contrário: a quarta é o dia de **maior** consumo (cerca de 38% da semana), e segunda e sexta ficam abaixo.

Olhando só o total por dia, parecia que todo mundo ouvia menos na quarta. Era Springfield puxando a média, porque tem mais volume.

[Voltar ao Índice](#back)

# Conclusões <a id='end'></a>

**A hipótese se confirma: a atividade dos usuários muda conforme o dia da semana e a cidade.**

- Springfield concentra cerca de 70% das reproduções.
- Os picos são diferentes: Springfield tem pico na segunda e na sexta, Shelbyville na quarta.
- O total por dia escondia esse padrão. Só apareceu quando separei por cidade.

**O que isso significa na prática**

Se eu fosse planejar uma campanha ou um lançamento nesse serviço, não usaria o mesmo calendário pras duas cidades. Em Springfield, o foco seria segunda e sexta. Em Shelbyville, quarta.

E a lição maior: **média geral pode enganar.** Quando um grupo é bem maior que o outro, ele dita o resultado. Sempre vale segmentar antes de decidir.

### Limitações

- Comparei volumes, sem teste estatístico. Um teste de significância deixaria a conclusão mais forte.
- A base tem só três dias da semana e uma única fonte de dados. Não dá pra generalizar para toda a população das cidades.

[Voltar ao Índice](#back)